## 1. Import Required Libraries

# MQTT to CSV Logger

This notebook subscribes to MQTT telemetry topics and appends readings to a CSV file. It supports multiple transport protocols (TCP, WebSockets) and optional Supabase integration.

**Installation:**
```
pip install paho-mqtt python-dotenv requests
```

**Configuration:**
Environment variables needed:
- `MQTT_URL`: MQTT broker URL (e.g., mqtt://host:1883 or mqtts://host:8883)
- `MQTT_USERNAME`: MQTT username
- `MQTT_PASSWORD`: MQTT password
- `MQTT_TOPIC_FILTER`: Topics to subscribe to (default: energy/+/+/telemetry)
- `SUPABASE_URL`: (optional) Supabase URL for data forwarding
- `SUPABASE_KEY`: (optional) Supabase API key

In [ ]:
from __future__ import annotations
import os
import csv
import json
import time
import signal
import argparse
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse

try:
    import paho.mqtt.client as mqtt
    from paho.mqtt.enums import CallbackAPIVersion
except Exception:
    raise SystemExit("Please install required package: pip install paho-mqtt")

try:
    import requests
except Exception:
    requests = None

try:
    from dotenv import load_dotenv
except ImportError:
    print('Warning: python-dotenv not installed. Install with: pip install python-dotenv')
    print('Falling back to system environment variables only.')
    load_dotenv = None

## 2. Set Environment Variables

In [ ]:
# Load environment variables from .env file
if load_dotenv:
    # Load from server/.env if it exists, otherwise try current directory
    env_path = Path('server') / '.env'
    if not env_path.exists():
        env_path = Path.cwd() / '.env'
    
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
        print(f"Loaded environment from: {env_path}")
    else:
        print(f"Warning: .env file not found at {env_path}")

# Get environment variables
MQTT_URL = os.environ.get('MQTT_URL')
MQTT_USERNAME = os.environ.get('MQTT_USERNAME')
MQTT_PASSWORD = os.environ.get('MQTT_PASSWORD')
MQTT_TOPIC_FILTER = os.environ.get('MQTT_TOPIC_FILTER', 'energy/+/+/telemetry')
SUPABASE_URL = os.environ.get('SUPABASE_URL') or os.environ.get('DB_SUPABASE_URL')
SUPABASE_KEY = os.environ.get('SUPABASE_KEY') or os.environ.get('DB_SUPABASE_KEY')

print("\nEnvironment Configuration:")
print(f"  MQTT_URL: {MQTT_URL}")
print(f"  MQTT_USERNAME: {MQTT_USERNAME if MQTT_USERNAME else 'Not set'}")
print(f"  MQTT_PASSWORD: {'*' * len(MQTT_PASSWORD) if MQTT_PASSWORD else 'Not set'}")
print(f"  MQTT_TOPIC_FILTER: {MQTT_TOPIC_FILTER}")
print(f"  SUPABASE_URL: {SUPABASE_URL if SUPABASE_URL else 'Not set'}")

## 3. Define MQTT Configuration and Constants

In [ ]:
# CSV field names
FIELDNAMES = [
    'ts', 'ts_iso', 'topic', 'device_type', 'device_id',
    'voltage', 'shunt_mV', 'current', 'power',
    'soc_percent', 'soh_percent', 'uptime_ms', 'raw_payload'
]

# Global flag for graceful shutdown
stop_requested = False

def signal_handler(sig, frame):
    """Handle SIGINT and SIGTERM for graceful shutdown"""
    global stop_requested
    stop_requested = True
    print("\nShutdown requested...")

# Register signal handlers
signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

## 4. Parse Broker URL

In [ ]:
def parse_broker_url(url: str):
    """
    Parse MQTT broker URL
    
    Returns:
        tuple: (host, port, scheme, path)
    
    Example:
        parse_broker_url('mqtt://localhost:1883') -> ('localhost', 1883, 'mqtt', '')
        parse_broker_url('mqtts://broker.example.com:8883') -> ('broker.example.com', 8883, 'mqtts', '')
    """
    if not url:
        return None, None, None, None
    
    p = urlparse(url)
    scheme = p.scheme
    host = p.hostname
    port = p.port
    path = p.path or ''
    
    return host, port, scheme, path


# Test the parse_broker_url function
test_url = "mqtts://broker.hivemq.cloud:8883"
host, port, scheme, path = parse_broker_url(test_url)
print(f"Parsed URL: {test_url}")
print(f"  Host: {host}, Port: {port}, Scheme: {scheme}, Path: {path}")

## 5. Implement MQTTToCSV Class

In [ ]:
class MQTTToCSV:
    """
    MQTT subscriber that logs telemetry data to CSV file
    
    Supports:
    - Multiple transport protocols (TCP, WebSockets)
    - TLS/SSL encryption (mqtts, wss schemes)
    - Custom topic filtering
    - Optional Supabase integration
    """
    
    def __init__(self, broker: Optional[str], port: Optional[int], scheme: Optional[str], 
                 path: Optional[str], username: Optional[str], password: Optional[str], 
                 topic: str, outfile: str, supabase_url: Optional[str] = None, 
                 supabase_key: Optional[str] = None):
        """Initialize MQTT client and CSV file"""
        self.broker = broker
        self.port = port
        self.scheme = scheme
        self.path = path or ''
        self.username = username
        self.password = password
        self.topic = topic
        self.outfile = Path(outfile)
        self.supabase_url = supabase_url
        self.supabase_key = supabase_key
        
        # Choose transport: websockets for ws/wss, otherwise tcp
        transport = 'websockets' if (scheme in ('ws', 'wss')) else 'tcp'
        self.client = mqtt.Client(callback_api_version=CallbackAPIVersion.VERSION2, transport=transport)
        self.client.on_connect = self.on_connect
        self.client.on_message = self.on_message
        
        if username:
            self.client.username_pw_set(username, password)
        
        # TLS for secure connections
        if scheme in ('mqtts', 'wss') or (self.port == 8883) or (self.port == 8884):
            try:
                self.client.tls_set()
                # For testing allow insecure; remove for production
                self.client.tls_insecure_set(True)
            except Exception as e:
                print(f"TLS setup warning: {e}")
        
        # If using websockets and a path, set ws options
        if scheme in ('ws', 'wss') and self.path:
            try:
                self.client.ws_set_options(path=self.path)
            except Exception as e:
                print(f"WebSocket path setup warning: {e}")
        
        # Ensure CSV exists with headers
        if not self.outfile.exists():
            with self.outfile.open('w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
                writer.writeheader()
            print(f"Created CSV file: {self.outfile}")
    
    def on_connect(self, client, userdata, flags, reason_code, properties=None):
        """Handle MQTT connection"""
        print(f'MQTT connected, rc={reason_code}')
        client.subscribe(self.topic)
        print(f'Subscribed to topic: {self.topic}')
    
    def on_message(self, client, userdata, msg):
        """Handle incoming MQTT messages"""
        try:
            payload_raw = msg.payload.decode('utf-8')
            try:
                payload = json.loads(payload_raw)
            except Exception:
                payload = payload_raw
            
            ts = int(time.time() * 1000)
            ts_iso = time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(ts / 1000.0))
            
            # Extract device_type and device_id from topic
            parts = msg.topic.split('/')
            device_type = ''
            device_id = ''
            if len(parts) >= 4 and parts[0] == 'energy':
                device_type = parts[1]
                device_id = parts[2]
            elif len(parts) >= 1 and parts[0] == 'battery':
                # ESP32 BMS publishes to battery/data
                device_type = 'battery'
                device_id = 'esp32_bms'
            
            p = payload if isinstance(payload, dict) else {}
            row = {
                'ts': ts,
                'ts_iso': ts_iso,
                'topic': msg.topic,
                'device_type': device_type,
                'device_id': device_id,
                'voltage': p.get('bus_V') or p.get('voltage', ''),
                'shunt_mV': p.get('shunt_mV', ''),
                'current': p.get('current_A') or p.get('current', ''),
                'power': p.get('power_W') or p.get('power', ''),
                'soc_percent': p.get('soc_percent', ''),
                'soh_percent': p.get('soh_percent', ''),
                'uptime_ms': p.get('uptime_ms', ''),
                'raw_payload': payload_raw
            }
            
            # Append row to CSV
            with self.outfile.open('a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
                writer.writerow(row)
            
            print(f'Saved: {device_type} {device_id} -> {self.outfile}')
            
            # Optionally forward to Supabase
            if self.supabase_url and self.supabase_key:
                try:
                    self._send_to_supabase(payload, row, msg.topic)
                except Exception as e:
                    print(f'Supabase upload failed: {e}')
        
        except Exception as e:
            print(f'Error processing message: {e}')
    
    def _send_to_supabase(self, payload, row, topic):
        """Send data to Supabase REST API"""
        if requests is None:
            raise RuntimeError('requests is required for Supabase uploads (pip install requests)')
        
        # Normalize payload
        if isinstance(payload, dict):
            p = payload
        else:
            try:
                p = json.loads(row.get('raw_payload') or '{}')
            except Exception:
                p = {}
        
        data = {
            'ts': row.get('ts'),
            'ts_iso': row.get('ts_iso'),
            'topic': topic,
            'device_type': row.get('device_type'),
            'device_id': row.get('device_id'),
            'voltage': p.get('bus_V') or p.get('voltage'),
            'current': p.get('current_A') or p.get('current'),
            'power': p.get('power_W') or p.get('power'),
            'soc': p.get('soc_percent') or p.get('soc'),
            'soh': p.get('soh_percent') or p.get('soh'),
            'uptime_ms': p.get('uptime_ms') or p.get('uptime'),
            'raw_payload': row.get('raw_payload')
        }
        
        url = self.supabase_url.rstrip('/') + '/rest/v1/telemetry'
        headers = {
            'apikey': self.supabase_key,
            'Authorization': f'Bearer {self.supabase_key}',
            'Content-Type': 'application/json',
            'Prefer': 'return=representation'
        }
        
        resp = requests.post(url, headers=headers, json=[data], timeout=10)
        if resp.status_code not in (200, 201):
            raise RuntimeError(f'Supabase insert failed: {resp.status_code} {resp.text}')
    
    def run(self):
        """Connect to broker and run event loop"""
        global stop_requested
        if not self.broker:
            raise RuntimeError('No broker host provided')
        
        host = self.broker
        port = self.port or (8883 if self.scheme in ('mqtts', 'wss') else 1883)
        
        print(f'Connecting to MQTT broker {host}:{port} (scheme={self.scheme}, path={self.path})')
        
        self.client.connect(host, port, keepalive=60)
        
        try:
            while not stop_requested:
                self.client.loop(timeout=1.0)
        except KeyboardInterrupt:
            stop_requested = True
            print("\nInterrupted by user (Ctrl+C) - shutting down...")
        finally:
            try:
                self.client.disconnect()
                print("Disconnected from broker")
            except Exception:
                pass

## 6. Execute Main Logic

In [ ]:
# Configuration - Modify these values or set environment variables
config = {
    'broker': MQTT_URL or 'mqtt://localhost:1883',
    'topic': MQTT_TOPIC_FILTER,
    'username': MQTT_USERNAME,
    'password': MQTT_PASSWORD,
    'outfile': 'Cycle 01/b1_discharge.csv',
    'supabase_url': SUPABASE_URL,
    'supabase_key': SUPABASE_KEY
}

print("\n=== MQTT to CSV Logger Configuration ===")
print(f"Broker: {config['broker']}")
print(f"Topic: {config['topic']}")
print(f"Output File: {config['outfile']}")
print(f"Supabase Integration: {'Enabled' if config['supabase_url'] else 'Disabled'}")
print("=========================================\n")

In [ ]:
# Parse broker URL and create service
broker_url = config['broker']
if '://' in broker_url:
    host, port, scheme, path = parse_broker_url(broker_url)
else:
    host = broker_url
    port = None
    scheme = None
    path = ''

# Ensure output directory exists
output_path = Path(config['outfile'])
output_path.parent.mkdir(parents=True, exist_ok=True)

# Create and run the MQTT to CSV service
service = MQTTToCSV(
    broker=host,
    port=port,
    scheme=scheme,
    path=path,
    username=config['username'],
    password=config['password'],
    topic=config['topic'],
    outfile=config['outfile'],
    supabase_url=config['supabase_url'],
    supabase_key=config['supabase_key']
)

print("Starting MQTT subscriber...")
print("(Press Ctrl+C to stop)\n")

try:
    service.run()
except KeyboardInterrupt:
    stop_requested = True
    print("\n\nShutdown requested (Ctrl+C). Waiting for cleanup...")
    try:
        service.client.disconnect()
        print("Disconnected from broker")
    except Exception:
        pass
except Exception as e:
    print(f"Error: {e}")

## 7. Data Analysis (Optional)

In [ ]:
# Read and display CSV data
import pandas as pd

# Use the same filename from config
outfile = Path(config['outfile'])
if outfile.exists():
    df = pd.read_csv(outfile)
    print(f"Data Summary ({len(df)} records):")
    print(f"Date Range: {df['ts_iso'].min()} to {df['ts_iso'].max()}")
    print(f"\nDevices: {df['device_id'].unique().tolist()}")
    print(f"\nTopics: {df['topic'].unique().tolist()}")
    print(f"\nLatest 5 records:")
    print(df.tail())
else:
    print(f"CSV file not found: {outfile}")
    print(f"Make sure to run cells 13-15 first to collect data.")